# GRPO Reinforcement Learning — Qwen2.5-1.5B on MetaMathQA

**Stack:** Unsloth + TRL GRPOTrainer + WandB  
**Base:** SFT checkpoint `Suryansh7123/qwen2.5_lora_r16_finetune`  
**LoRA:** r=4, alpha=8, rsLoRA (RL phase — lighter than SFT r=16)  
**Curriculum:** Easy (GSM ≤5 steps) → Medium (GSM 6-9 steps + simple MATH) → Hard (MATH-derived 9+ steps)  
**Reward:** correctness (0.8) + format (0.1) + length (0.1) + step structure bonus  

## Cell 1 — Install (pinned to working SFT versions)

**WHY these exact pins:**
- `unsloth[kaggle-new]` from git: always gets latest Unsloth which self-manages its own compatibility
- `--no-deps` on trl/peft/accelerate: prevents pip from upgrading transformers past 5.5.x which breaks Unsloth's patching
- Same pattern that eliminated PicklingError in your SFT notebook

In [1]:
# ── 1. Install — pinned to working SFT stack ──────────────────────────────────
# IDENTICAL install strategy to your working SFT notebook.
# --no-deps on trl/peft/accelerate = transformers stays at ~5.5.0
# which is the version Unsloth's kernel patches are validated against.
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ke1gi1fe/unsloth_2ad52cc5df074110a1764e297daaf3a5
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ke1gi1fe/unsloth_2ad52cc5df074110a1764e297daaf3a5
  Resolved https://github.com/unslothai/unsloth.git to commit 76a2b9edf160d68208dc30c02c6523bc6551f950
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.4 M

## Cell 2 — Clear compiled cache + disable dynamo

Stale Unsloth compiled kernels from a previous session cause silent wrong-dtype errors.
TorchDynamo disabled because it conflicts with Unsloth's custom CUDA ops on Kaggle's T4.

In [2]:
# ── 2. Clear stale cache + disable dynamo ─────────────────────────────────────
import shutil, os

cache_path = "/kaggle/working/unsloth_compiled_cache"
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Stale Unsloth cache cleared.")
else:
    print("No stale cache found — clean start.")

# Must be set BEFORE any torch/unsloth import
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"]      = "1"
print("Env vars set.")

No stale cache found — clean start.
Env vars set.


## Cell 3 — Imports

**Critical import order:** Unsloth BEFORE transformers — Unsloth patches torch ops in-place at import time.
Reversing this order causes unfused kernels and wastes VRAM.

In [3]:
# ── 3. Imports ────────────────────────────────────────────────────────────────
# ORDER MATTERS: unsloth before transformers (kernel patching)
import os, json, re, gc, time, math, random, collections
from typing import List, Dict, Optional

import numpy as np
import torch

# Unsloth FIRST
from unsloth import FastLanguageModel

import transformers
from transformers import TrainerCallback, TrainerState, TrainerControl
from datasets import load_dataset, Dataset
from trl import GRPOTrainer, GRPOConfig
import wandb
from huggingface_hub import login

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    cc  = f"{dev.major}.{dev.minor}"
    print(f"GPU          : {dev.name}")
    print(f"VRAM         : {dev.total_memory / 1e9:.1f} GB")
    print(f"Compute Cap  : {cc}")
    if float(cc) < 7.5:
        raise RuntimeError("Need CC >= 7.5 (T4 or better)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch        : 2.10.0+cu128
Transformers : 5.5.0
CUDA         : True
GPU          : Tesla T4
VRAM         : 15.6 GB
Compute Cap  : 7.5


## Cell 4 — Config

**Key differences from SFT config explained in comments.**

In [4]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    # ── Model: load YOUR SFT checkpoint, not the raw base ─────────────────────
    # This is a LoRA adapter repo. Unsloth loads the base weights + merges
    # the SFT adapter automatically when you pass the adapter repo path.
    model_name        = "Suryansh7123/qwen2.5_lora_r16_finetune",
    max_seq_len       = 1024,          # shorter than SFT (2048) — GRPO generates
                                      # G=6 completions per step, needs headroom

    # ── LoRA: r=4 for RL phase ────────────────────────────────────────────────
    # RL doesn't inject new knowledge, just reshapes output distribution.
    # r=4 = ~4x fewer params than SFT r=16 → less VRAM, less forgetting risk.
    lora_r            = 4,
    lora_alpha        = 8,            # alpha = 2*r → effective scale = sqrt(2)
    lora_dropout      = 0.0,          # 0 dropout for RL — cleaner gradient signal
    lora_target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias              = "none",
    use_rslora        = True,
        # ── Dataset filtering (NEW) ──────────────────────────────────────────────
    gsm_min_steps     = 9,          # Only take GSM problems with 7 or more steps
    gsm_max_steps     = 15,          # Cap at 9 to avoid excessively long ones
    medium_math_frac  = 0.0,        # REDUCED from 0.3 to 0.1 (or set to 0.0)

    # ── Dataset ───────────────────────────────────────────────────────────────
    dataset_name      = "meta-math/MetaMathQA",
    easy_size         = 0,         # GSM-derived, <= 5 steps
    medium_size       = 2000,         # GSM 6-9 steps + simpler MATH-derived (<= 9 steps)
    hard_size         = 0,         # MATH-derived, 9+ steps (hardest problems)
             # 30% of medium comes from MATH-derived source
    seed              = 42,

    # ── GRPO hyperparams ──────────────────────────────────────────────────────
    grpo_group_size   = 6,            # G — rollouts per prompt
    max_new_tokens    = 500,          # generation cap per rollout
    temperature       = 0.8,          # some diversity needed for GRPO signal
    kl_coeff          = 0.03,         # beta — low because SFT model is already good

    # ── Reward weights ────────────────────────────────────────────────────────
    w_correct         = 0.80,         # binary: answer matches ground truth
    w_format          = 0.10,         # "The answer is:" present in output
    w_length          = 0.10,         # penalty only above 400 tokens
    length_threshold  = 300,          # tokens above this get penalised
    step_bonus        = 0.05,         # bonus for multi-line reasoning

    # ── Curriculum ────────────────────────────────────────────────────────────
    curriculum_window = 200,          # rolling window size (problems)
    curriculum_check  = 50,           # check every N steps
    curriculum_thresh = 0.70,         # 70% pass@1 accuracy to promote
    # Tiers in order: easy -> medium -> hard

    # ── Training ──────────────────────────────────────────────────────────────
    output_dir        = "/kaggle/working/qwen1b5_grpo_rl",
    per_device_batch  = 12,            # GRPO: 1 prompt per step (G=6 rollouts)
    grad_accum        = 4,            # effective batch = 4 prompts
    learning_rate     = 5e-7,         # much lower than SFT — RL needs tiny steps
    num_train_epochs  = 1,
    max_grad_norm     = 0.1,          # tighter clip for RL stability

    # T4 precision — same as your working SFT notebook
    fp16              = True,
    bf16              = False,

    # ── Logging / saving ──────────────────────────────────────────────────────
    logging_steps     = 1,
    save_steps        = 100,
    save_total_limit  = 2,
    print_sample_every= 25,           # print model output + rewards every N steps
    run_name          = "qwen1b5-grpo-metamath-r4",
)

os.makedirs(CFG["output_dir"], exist_ok=True)
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

{
  "model_name": "Suryansh7123/qwen2.5_lora_r16_finetune",
  "max_seq_len": "1024",
  "lora_r": "4",
  "lora_alpha": "8",
  "lora_dropout": "0.0",
  "lora_target_modules": "['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']",
  "bias": "none",
  "use_rslora": "True",
  "gsm_min_steps": "9",
  "gsm_max_steps": "15",
  "medium_math_frac": "0.0",
  "dataset_name": "meta-math/MetaMathQA",
  "easy_size": "0",
  "medium_size": "2000",
  "hard_size": "0",
  "seed": "42",
  "grpo_group_size": "6",
  "max_new_tokens": "500",
  "temperature": "0.8",
  "kl_coeff": "0.03",
  "w_correct": "0.8",
  "w_format": "0.1",
  "w_length": "0.1",
  "length_threshold": "400",
  "step_bonus": "0.05",
  "curriculum_window": "200",
  "curriculum_check": "50",
  "curriculum_thresh": "0.7",
  "output_dir": "/kaggle/working/qwen1b5_grpo_rl",
  "per_device_batch": "2",
  "grad_accum": "4",
  "learning_rate": "5e-07",
  "num_train_epochs": "1",
  "max_grad_norm": "0.1",
  "fp16": "True",
 

## Cell 5 — HuggingFace + WandB login

In [5]:
# ── 5. Auth ───────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# HF login — needed to load your private LoRA adapter
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("HF Hub: logged in")

HF_RL_REPO = "Suryansh7123/qwen2.5_grpo_rl_r4_metamath"   # new repo for RL checkpoint
print(f"RL checkpoints will push to: {HF_RL_REPO}")



print("WandB: run initialised")

HF Hub: logged in
RL checkpoints will push to: Suryansh7123/qwen2.5_grpo_rl_r4
WandB: run initialised


## Cell 6 — Load SFT model + inject fresh RL LoRA

**Why `load_in_4bit=False, dtype=torch.float16`:**  
Loading an adapter repo with `load_in_4bit=True` attaches the SFT LoRA to the base model as a PEFT layer.
Calling `get_peft_model()` immediately after then raises `TypeError: model already has LoRA adapters`
because bnb refuses to stack a second LoRA config on top. The fix is `load_in_4bit=False` —
Unsloth auto-merges the SFT adapter into the base weights on load, returning a clean dense fp16 model.
`get_peft_model()` can then inject the new RL LoRA without conflict.  
VRAM cost: fp16 dense Qwen2.5-1.5B ≈ 3 GB. With G=6 rollouts on a 15 GB T4 this is fine.  
Do not revert to `load_in_4bit=True` — it re-introduces both the LoRA conflict and the dtype mismatch.

In [6]:
# ── 6. Load SFT checkpoint + merge SFT LoRA → inject new RL LoRA ─────────────
print("Loading SFT model via FastLanguageModel...")
print(f"  Source : {CFG['model_name']}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = CFG["model_name"],
    max_seq_length= CFG["max_seq_len"],
    dtype         = torch.float16,
    load_in_4bit  = False,
    token         = hf_token,
)

# Unsloth 2026.6.x no longer auto-merges on load_in_4bit=False.
# Check if SFT LoRA is still attached as live PEFT layers and merge explicitly.
from peft import PeftModel
if isinstance(model, PeftModel):
    print("SFT LoRA still attached — merging explicitly...")
    model = model.merge_and_unload()
    print(f"Merged. Dtype: {next(model.parameters()).dtype}")
else:
    print("SFT LoRA already merged on load.")

import gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM after merge : {torch.cuda.memory_reserved()/1e9:.2f} GB")

# Qwen uses eos as pad — must be set explicitly
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Inject fresh RL LoRA ──────────────────────────────────────────────────────
print("\nInjecting NEW RL LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r                          = CFG["lora_r"],
    target_modules             = CFG["lora_target_modules"],
    lora_alpha                 = CFG["lora_alpha"],
    lora_dropout               = CFG["lora_dropout"],
    bias                       = CFG["bias"],
    use_rslora                 = CFG["use_rslora"],
    use_gradient_checkpointing = "unsloth",
    random_state               = CFG["seed"],
)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params     : {total_p/1e6:.1f} M")
print(f"Trainable params : {trainable_p/1e6:.1f} M  ({100*trainable_p/total_p:.2f}%)")
print(f"VRAM after LoRA  : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"\nLoRA r={CFG['lora_r']}, alpha={CFG['lora_alpha']}, rsLoRA={CFG['use_rslora']}")

Loading SFT model via FastLanguageModel...
  Source : Suryansh7123/qwen2.5_lora_r16_finetune
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

Unsloth 2026.6.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


SFT LoRA still attached — merging explicitly...
Merged. Dtype: torch.float16
VRAM after merge : 3.17 GB

Injecting NEW RL LoRA adapters...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Unsloth 2026.6.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



Total params     : 1548.3 M
Trainable params : 4.6 M  (0.30%)
VRAM after LoRA  : 3.19 GB

LoRA r=4, alpha=8, rsLoRA=True


## Cell 7 — Prompt template

**Same Alpaca template as SFT.** Changing this would break lm_eval GSM8K extraction
since lm_eval uses the same template to format prompts at evaluation time.

In [7]:
# ── 7. Prompt template — identical to SFT notebook ────────────────────────────
ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n### Response:"
)

def make_prompt(question: str) -> str:
    return ALPACA_TEMPLATE.format(instruction=question.strip())

# Sanity check
print(make_prompt("What is 2+2?"))
print("\nTemplate OK")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is 2+2?

### Response:

Template OK


## Cell 8 — Dataset: load + 3-tier curriculum split

**Type field values in MetaMathQA:**
- GSM-derived: `GSM_SV`, `GSM_Rephrased`, `GSM_AnsAug`, `GSM_FOBAR`, `GSM_MATHS-Rephrased`
- MATH-derived: `MATH_AnsAug`, `MATH_Rephrased`, `MATH_SV`, `MATH_FOBAR`

**Three-tier split logic:**
- **Easy:** GSM-derived AND ≤ 5 reasoning lines — elementary arithmetic, single-variable problems
- **Medium:** GSM-derived with 6-9 lines (multi-step word problems) + 30% MATH-derived with ≤ 9 lines (simpler algebra)
- **Hard:** MATH-derived with 9+ lines — combinatorics, geometry, number theory, advanced algebra

**Why 1.5k per tier:** At ~30s/step on T4 with G=6, one full pass through 1.5k samples = ~12.5 hours.
Each tier fits exactly in one Kaggle session, making multi-session chaining clean and predictable.

In [8]:
# ── 8. Dataset — 3-tier curriculum split ─────────────────────────────────────
print("Loading MetaMathQA...")
raw_ds = load_dataset(CFG["dataset_name"], split="train")
print(f"Total samples : {len(raw_ds)}")
print(f"Columns       : {raw_ds.column_names}")

# ── Helper functions ─────────────────────────────────────────────────────────
def count_steps(response: str) -> int:
    """Estimate reasoning depth by counting non-empty lines in the GT response.
    GT response lines ≈ reasoning steps — more reliable than character count."""
    return len([l for l in response.split("\n") if l.strip()])

def extract_answer(text: str) -> Optional[str]:
    """Extract final numeric answer. Tries MetaMathQA format, then GSM8K, then last number."""
    m = re.search(r"[Tt]he answer is:?\s*([\-\d,\.]+)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    m = re.search(r"####\s*([\-\d,\.]+)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"[\-]?\d+\.?\d*", text)
    return nums[-1] if nums else None

def answers_equal(pred: str, gold: str) -> bool:
    """Numeric equality with float tolerance."""
    try:
        return abs(float(pred) - float(gold)) < 1e-4
    except (ValueError, TypeError):
        return pred.strip() == gold.strip()

# ── Type prefix sets ─────────────────────────────────────────────────────────
# All known GSM-derived type prefixes in MetaMathQA
GSM_PREFIXES  = ("GSM_SV", "GSM_Rephrased", "GSM_AnsAug", "GSM_FOBAR", "GSM_MATHS-Rephrased")
# All known MATH-derived type prefixes
MATH_PREFIXES = ("MATH_SV", "MATH_Rephrased", "MATH_AnsAug", "MATH_FOBAR")

# ── Step thresholds ───────────────────────────────────────────────────────────
EASY_MAX   = 5   # GSM-derived, <= 5 lines
MEDIUM_MAX = 15  # GSM 6-9 lines  OR  MATH-derived <= 9 lines
# Hard = MATH-derived with > 9 lines

# ── Scan full dataset into buckets ────────────────────────────────────────────
easy_samples   = []
medium_samples = []   # two sub-sources — GSM hard-ish + simple MATH
hard_samples   = []

medium_gsm_pool  = []   # GSM 6-9 steps
medium_math_pool = []   # MATH-derived <= 9 steps

for ex in raw_ds:
    q    = ex["query"]
    r    = ex["response"]
    typ  = ex["type"]
    ans  = extract_answer(r)
    if ans is None:
        continue   # can't compute reward without GT answer

    steps = count_steps(r)
    rec   = {"question": q, "answer": ans, "type": typ}

    is_gsm  = any(typ.startswith(p) for p in GSM_PREFIXES)
    is_math = any(typ.startswith(p) for p in MATH_PREFIXES)

    if is_gsm:
        if steps <= EASY_MAX:
            easy_samples.append(rec)   # We will discard these later since easy_size=0
        # NEW: Only take GSM problems that fall into your 7-9 step "tough" bracket
        elif steps >= CFG["gsm_min_steps"] and steps <= CFG["gsm_max_steps"]:
            medium_gsm_pool.append(rec)
        # GSM with <7 or >9 steps: drop entirely
# Replace the print section at the bottom of Cell 8 with this:

# ── Build medium tier: 70% GSM-hard + 30% simple MATH ────────────────────────
# medium_math_frac = 0.3 set in CFG
random.shuffle(medium_gsm_pool)
random.shuffle(medium_math_pool)
random.shuffle(easy_samples)
random.shuffle(hard_samples)

n_med        = CFG["medium_size"]
n_math_med   = int(n_med * CFG["medium_math_frac"])    # 30% from MATH
n_gsm_med    = n_med - n_math_med                      # 70% from GSM
medium_samples = medium_gsm_pool[:n_gsm_med] + medium_math_pool[:n_math_med]
random.shuffle(medium_samples)   # shuffle to interleave the two sources

# ── Final subsampled datasets ─────────────────────────────────────────────────
easy_samples = easy_samples[:CFG["easy_size"]]
hard_samples = hard_samples[:CFG["hard_size"]]

easy_ds   = Dataset.from_list(easy_samples)
medium_ds = Dataset.from_list(medium_samples)
hard_ds   = Dataset.from_list(hard_samples)

# ── CRITICAL: add 'prompt' column ────────────────────────────────────────────
# GRPOTrainer internally does x["prompt"] on every sample.
# Without this column the trainer raises KeyError: 'prompt' immediately.
# 'answer' column must also be preserved — reward function reads it via **kwargs.
easy_ds   = easy_ds.map(lambda ex: {"prompt": make_prompt(ex["question"])})
medium_ds = medium_ds.map(lambda ex: {"prompt": make_prompt(ex["question"])})
hard_ds   = hard_ds.map(lambda ex: {"prompt": make_prompt(ex["question"])})

print(f"\nFinal tier sizes:")

print(f"\nColumns: {easy_ds.column_names}")
print(f"\nColumns: {medium_ds.column_names if len(medium_ds) > 0 else easy_ds.column_names}" )

if len(easy_ds) > 0:
    print(f"\nSample easy   — Q: {easy_ds[0]['question'][:80]}")
    print(f"              — A: {easy_ds[0]['answer']}")
else:
    print("\nSample easy   — [No samples in Easy Tier]")

if len(medium_ds) > 0:
    print(f"Sample medium — Q: {medium_ds[0]['question'][:80]}")
    print(f"              — A: {medium_ds[0]['answer']} type={medium_ds[0]['type']}")
else:
    print("Sample medium — [No samples in Medium Tier]")

if len(hard_ds) > 0:
    print(f"Sample hard   — Q: {hard_ds[0]['question'][:80]}")
    print(f"              — A: {hard_ds[0]['answer']}")
else:
    print("Sample hard   — [No samples in Hard Tier]")


Loading MetaMathQA...


README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

Total samples : 395000
Columns       : ['type', 'query', 'original_question', 'response']


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Final tier sizes:

Columns: []

Columns: ['question', 'answer', 'type', 'prompt']

Sample easy   — [No samples in Easy Tier]
Sample medium — Q: If the mayor paid a total of $50,000 to two artists for painting 50 murals aroun
              — A: 9800 type=GSM_Rephrased
Sample hard   — [No samples in Hard Tier]


## Cell 9 — Reward function

**Four components:**
1. **Correctness (0.80):** Binary. Extract last number from output, compare to GT after float normalization.
2. **Format (0.10):** Does output contain `The answer is:` ? Keeps output compatible with lm_eval's GSM8K extractor.
3. **Length penalty (0.10):** Zero below 400 tokens. Linear penalty above — prevents reward hacking via verbose outputs.
4. **Step structure bonus (0.05 additive):** Bonus if output has 3+ lines before the answer. Prevents the degenerate policy of just outputting the answer token directly.

**GRPO API contract:** reward function receives `(prompts, completions, **kwargs)` and returns `List[float]`.

In [9]:
# ── 9. Reward function ────────────────────────────────────────────────────────

# Ground-truth answers are passed via dataset — we read them from a closure
# The GRPOTrainer passes dataset fields as kwargs if they're in the dataset

def compute_rewards(
    prompts: List[str],
    completions: List[str],
    answer: List[str],          # GT answer — passed from dataset column
    **kwargs
) -> List[float]:
    """
    Returns one reward per completion.
    GRPO groups G completions per prompt, so len(completions) = len(prompts) * G.
    answer is repeated G times per prompt by GRPOTrainer.
    """
    rewards = []

    for completion, gt_answer in zip(completions, answer):
        reward = 0.0

        # ── 1. Correctness (weight 0.80) ──────────────────────────────────────
        pred_answer = extract_answer(completion)
        if pred_answer is not None and answers_equal(pred_answer, gt_answer):
            correctness = 1.0
        else:
            correctness = 0.0
        reward += CFG["w_correct"] * correctness

        # ── 2. Format compliance (weight 0.10) ────────────────────────────────
        # Check that model outputs in a format lm_eval can extract
        has_format = 1.0 if re.search(r"[Tt]he answer is:?", completion) else 0.0
        reward += CFG["w_format"] * has_format

        # ── 3. Length penalty (weight 0.10) ───────────────────────────────────
        # Zero penalty under threshold, linear penalty above
        token_count = len(completion.split())   # word approximation, fast
        if token_count <= CFG["length_threshold"]:
            length_score = 1.0
        else:
            overage = token_count - CFG["length_threshold"]
            # -0.1 per 50 extra words, floor at 0
            length_score = max(0.0, 1.0 - (overage / 50) * 0.1)
        reward += CFG["w_length"] * length_score

        # ── 4. Step structure bonus (additive 0.05) ───────────────────────────
        # Prevents degenerate policy of outputting just the answer number.
        # Count non-empty lines before the answer line.
        lines = [l.strip() for l in completion.split("\n") if l.strip()]
        answer_line_idx = len(lines)  # default: answer is last
        for idx, line in enumerate(lines):
            if re.search(r"[Tt]he answer is:?", line):
                answer_line_idx = idx
                break
        reasoning_lines = answer_line_idx
        if reasoning_lines >= 3:
            reward += CFG["step_bonus"]

        rewards.append(reward)

    return rewards

# Quick sanity check
test_comps = [
    "First I add 3 + 4 = 7.\nThen multiply by 2 = 14.\nSo the total is 14.\nThe answer is: 14",
    "The answer is: 99",
    "I think it might be 14 or maybe 15, hard to say really and there are many ways to approach this kind of problem...",
]
test_gt = ["14", "14", "14"]
test_rewards = compute_rewards(
    prompts=["q"] * 3, completions=test_comps, answer=test_gt
)
print("Reward function sanity check:")
for c, r in zip(test_comps, test_rewards):
    print(f"  reward={r:.3f}  |  '{c[:60]}...'")

# Expected: ~0.95 (correct+format+length+step), ~0.80 (wrong but has format), ~0.05 (wrong+verbose)

Reward function sanity check:
  reward=1.050  |  'First I add 3 + 4 = 7.
Then multiply by 2 = 14.
So the total...'
  reward=0.200  |  'The answer is: 99...'
  reward=0.100  |  'I think it might be 14 or maybe 15, hard to say really and t...'


## Cell 10 — Curriculum tracker + monitoring callback

**Curriculum logic (3 tiers):**  
Maintains a rolling deque of the last 200 pass@1 outcomes. Checks every 50 steps.  
- 70% accuracy on **easy** → promote to **medium** (reset window)  
- 70% accuracy on **medium** → promote to **hard** (reset window)  
Promotion is a one-way gate — no demotion. Tier is stored as `curriculum.tier` ∈ {`'easy'`, `'medium'`, `'hard'`}.  
WandB logs tier as integer (0/1/2) so you can see the step where promotion happened on the graph.

**Monitoring callback:**  
Every `print_sample_every` steps: generates one greedy sample from the active tier dataset,  
prints full model output + per-component reward breakdown to stdout and logs to WandB.

In [10]:
# ── 10. Curriculum tracker + monitoring callback ──────────────────────────────

TIER_ORDER = ["easy", "medium", "hard"]   # fixed promotion sequence
TIER_INT   = {"easy": 0, "medium": 1, "hard": 2}   # for WandB numeric logging

class CurriculumState:
    """3-tier rolling accuracy tracker. Promotes easy → medium → hard."""

    def __init__(self, window: int, threshold: float, check_every: int):
        self.window      = window
        self.threshold   = threshold
        self.check_every = check_every
        self.outcomes    = collections.deque(maxlen=window)
        self.tier        = "easy"
        self.promoted_at = {}   # {tier_name: step}

    def record(self, correct: bool):
        self.outcomes.append(1 if correct else 0)

    def rolling_accuracy(self) -> float:
        if not self.outcomes:
            return 0.0
        return sum(self.outcomes) / len(self.outcomes)

    def next_tier(self) -> Optional[str]:
        idx = TIER_ORDER.index(self.tier)
        return TIER_ORDER[idx + 1] if idx + 1 < len(TIER_ORDER) else None

    def should_promote(self, step: int) -> bool:
        if self.next_tier() is None:
            return False   # already at hard, nowhere to go
        if step % self.check_every != 0:
            return False
        if len(self.outcomes) < self.window // 2:
            return False   # need at least half the window filled
        return self.rolling_accuracy() >= self.threshold

    def promote(self, step: int):
        old_tier = self.tier
        self.tier = self.next_tier()
        self.promoted_at[old_tier] = step
        self.outcomes.clear()   # reset window — fresh tracking on new tier
        print(f"\n{'='*65}")
        print(f"  CURRICULUM PROMOTION at step {step}")
        print(f"  Rolling accuracy exceeded {self.threshold*100:.0f}% on {old_tier.upper()} tier.")
        print(f"  Advancing to {self.tier.upper()} tier.")
        print(f"  ACTION: stop training cell and run Cell 14 ({self.tier} trainer).")
        print(f"{'='*65}\n")


curriculum = CurriculumState(
    window      = CFG["curriculum_window"],
    threshold   = CFG["curriculum_thresh"],
    check_every = CFG["curriculum_check"],
)


class GRPOMonitorCallback(TrainerCallback):
    """
    Every `print_sample_every` steps:
      - Generates one greedy sample from active tier dataset
      - Prints question / model output / per-component reward breakdown
      - Logs curriculum tier (0/1/2) + rolling accuracy to WandB
    Also records pass@1 outcomes for curriculum promotion logic.
    """

    def __init__(self, tokenizer, model, dataset_ref: dict, curriculum: CurriculumState):
        self.tokenizer   = tokenizer
        self.model       = model
        # dataset_ref keys must match TIER_ORDER values
        self.ds_ref      = dataset_ref   # {'easy': easy_ds, 'medium': medium_ds, 'hard': hard_ds}
        self.curriculum  = curriculum
        self.step        = 0
        self._sample_idx = 0

    def on_log(self, args, state: TrainerState, control: TrainerControl, logs=None, **kwargs):
        self.step = state.global_step
        if logs is None:
            return

        # ── Record pass@1 from mean group reward ──────────────────────────────
        mean_reward = logs.get("reward", None)
        if mean_reward is not None:
            self.curriculum.record(mean_reward > CFG["w_correct"] * 0.5)

        # ── Curriculum promotion check ────────────────────────────────────────
        if self.curriculum.should_promote(self.step):
            self.curriculum.promote(self.step)



        # ── Print sample every N steps ────────────────────────────────────────
        if self.step > 0 and self.step % CFG["print_sample_every"] == 0:
            self._print_sample()
    def _print_sample(self):
        active_ds = self.ds_ref[self.curriculum.tier]
        if len(active_ds) == 0:
            print(f"WARNING: {self.curriculum.tier} dataset is empty, skipping sample print.")
            return   # <-- Add this
        idx = self._sample_idx % len(active_ds)
        """Generate one greedy sample and print full reward breakdown."""
        active_ds = self.ds_ref[self.curriculum.tier]
        idx       = self._sample_idx % len(active_ds)
        ex        = active_ds[idx]
        self._sample_idx += 1

        q      = ex["question"]
        gt     = ex["answer"]
        prompt = make_prompt(q)

        FastLanguageModel.for_inference(self.model)
        enc = self.tokenizer(
            prompt, return_tensors="pt", truncation=True,
            max_length=CFG["max_seq_len"] // 2
        ).to(self.model.device)

        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens = CFG["max_new_tokens"],
                do_sample      = False,
                temperature    = 1.0,
                pad_token_id   = self.tokenizer.pad_token_id,
                eos_token_id   = self.tokenizer.eos_token_id,
                use_cache      = True,
            )

        response = self.tokenizer.decode(
            out[0][enc.input_ids.shape[1]:], skip_special_tokens=True
        )
        FastLanguageModel.for_training(self.model)

        rewards   = compute_rewards(prompts=[prompt], completions=[response], answer=[gt])
        total_r   = rewards[0]
        pred_ans  = extract_answer(response)
        correct   = pred_ans is not None and answers_equal(pred_ans, gt)
        has_fmt   = bool(re.search(r"[Tt]he answer is:?", response))
        token_cnt = len(response.split())
        lines_bf  = len([l for l in response.split("\n") if l.strip()])
        rolling_acc = self.curriculum.rolling_accuracy()

        sep = "═" * 65
        print(f"\n{sep}")
        print(f"  STEP {self.step} — {self.curriculum.tier.upper()} TIER SAMPLE")
        print(f"  Rolling accuracy : {rolling_acc*100:.1f}%  "
              f"(window={len(self.curriculum.outcomes)}/{self.curriculum.window})  "
              f"promote at {CFG['curriculum_thresh']*100:.0f}%")
        print(f"  Next tier        : {self.curriculum.next_tier() or 'none (already at hard)'}")
        print(sep)
        print(f"  QUESTION : {q[:120]}")
        print(f"  GT ANSWER: {gt}")
        print(f"  ─── MODEL OUTPUT ({token_cnt} words) ───")
        print(f"  {response[:500]}")
        if len(response) > 500:
            print(f"  ... [truncated, full={len(response)} chars]")
        print(f"  ─── REWARD BREAKDOWN ───")
        print(f"  Predicted answer : {pred_ans}")
        print(f"  Correct          : {correct}  "
              f"(w={CFG['w_correct']:.2f} → {CFG['w_correct']*float(correct):.3f})")
        print(f"  Format OK        : {has_fmt}  "
              f"(w={CFG['w_format']:.2f} → {CFG['w_format']*float(has_fmt):.3f})")
        print(f"  Length           : {token_cnt} words (threshold={CFG['length_threshold']})")
        print(f"  Reasoning lines  : {lines_bf}  → step bonus: {lines_bf >= 3}")
        print(f"  TOTAL REWARD     : {total_r:.4f}")
        print(sep + "\n")



print("Curriculum (3-tier) and callback classes defined.")
print(f"Tier order: {' → '.join(TIER_ORDER)}")

Curriculum (3-tier) and callback classes defined.
Tier order: easy → medium → hard


## Cell 11 — Smoke test

Runs before training. Checks:
1. Library versions match working SFT stack
2. GPU + VRAM
3. Dataset slices and answer extraction
4. Reward function: non-degenerate (not all 0 or all 1)
5. Full pipeline: prompt → G=6 rollouts → reward for all 6
6. VRAM budget after model load

**This is the most important cell.** If it passes, `trainer.train()` will not crash on setup issues.

## Cell 12 — GRPOTrainer setup

**Why `num_generations=grpo_group_size`:** This is the G in GRPO. Each prompt gets G completions,
rewards are computed for all G, advantage is normalized within the group.  

**Why `optim='adamw_8bit'`:** Same as SFT — saves ~500MB on optimizer states.  

**KL coeff 0.01:** Low because SFT model is already good. Higher KL would prevent RL from moving the distribution enough to learn. Lower KL risks reward hacking.

In [11]:
# ── 12. GRPOTrainer setup ─────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# Session 1 always starts on easy tier.
# After promotion to medium, stop this cell and run Cell 14a.
# After promotion to hard, stop Cell 14a and run Cell 14b.
active_dataset = easy_ds
print(f"Starting with {curriculum.tier.upper()} tier: {len(active_dataset)} samples")

monitor_callback = GRPOMonitorCallback(
    tokenizer   = tokenizer,
    model       = model,
    dataset_ref = {"easy": easy_ds, "medium": medium_ds, "hard": hard_ds},
    curriculum  = curriculum,
)

grpo_config = GRPOConfig(
    output_dir                   = CFG["output_dir"],

    # GRPO-specific
    num_generations              = CFG["grpo_group_size"],   # G=6 rollouts per prompt
    max_completion_length        = CFG["max_new_tokens"],
    beta                         = CFG["kl_coeff"],          # KL penalty coefficient
    temperature                  = CFG["temperature"],

    # Training
    num_train_epochs             = CFG["num_train_epochs"],
    per_device_train_batch_size  = CFG["per_device_batch"],
    gradient_accumulation_steps  = CFG["grad_accum"],
    learning_rate                = CFG["learning_rate"],
    max_grad_norm                = CFG["max_grad_norm"],
    warmup_steps                 = 20,
    lr_scheduler_type            = "cosine",

    # Precision — T4 safe (same as SFT notebook)
    fp16                         = CFG["fp16"],
    bf16                         = CFG["bf16"],

    optim                        = "adamw_8bit",
    gradient_checkpointing       = True,
    gradient_checkpointing_kwargs= {"use_reentrant": False},



    # Saving + Hub
    save_strategy                = "steps",
    save_steps                   = CFG["save_steps"],
    save_total_limit             = CFG["save_total_limit"],
    push_to_hub                  = True,
    hub_model_id                 = HF_RL_REPO,
    hub_strategy                 = "checkpoint",

    seed                         = CFG["seed"],
    remove_unused_columns        = False,  # CRITICAL: keep 'answer' column for reward fn
)

trainer = GRPOTrainer(
    model             = model,
    tokenizer         = tokenizer,
    reward_funcs      = compute_rewards,
    args              = grpo_config,
    train_dataset     = active_dataset,
    callbacks         = [monitor_callback],
)

print("GRPOTrainer created OK")
print(f"Train dataset: {len(trainer.train_dataset)} samples")
print(f"Steps per epoch (approx): {len(trainer.train_dataset) // CFG['per_device_batch']}")
print(f"VRAM now: {torch.cuda.memory_reserved()/1e9:.2f} GB")

Starting with EASY tier: 0 samples
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 6
GRPOTrainer created OK
Train dataset: 0 samples
Steps per epoch (approx): 0
VRAM now: 3.19 GB


## Cell 13 — Train

## Cell 14a — Medium tier (run after easy promotes)

When the monitoring banner shows `CURRICULUM PROMOTION → MEDIUM`, stop Cell 13 and run this cell.  
Lower LR than easy (model already warm), same KL coeff.

In [12]:
# ── 14a. Medium tier training ─────────────────────────────────────────────────
# Run when curriculum.tier == 'medium' (callback printed promotion banner)
print(f"Curriculum tier is: {curriculum.tier}")

gc.collect(); torch.cuda.empty_cache()

medium_trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = compute_rewards,
    args          = GRPOConfig(
        output_dir                   = CFG["output_dir"] + "_medium",
        num_generations              = CFG["grpo_group_size"],
        max_completion_length        = CFG["max_new_tokens"],
        beta                         = CFG["kl_coeff"],
        temperature                  = CFG["temperature"],
        num_train_epochs             = 1,
        per_device_train_batch_size  = CFG["per_device_batch"],
        gradient_accumulation_steps  = CFG["grad_accum"],
        learning_rate                = CFG["learning_rate"] * ,  # slightly lower than easy
        max_grad_norm                = CFG["max_grad_norm"],
        warmup_steps                 = 10,
        lr_scheduler_type            = "cosine",
        fp16                         = CFG["fp16"],
        bf16                         = CFG["bf16"],
        optim                        = "adamw_8bit",
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        logging_steps                = 1,
        logging_first_step           = True,
        report_to                    = "none",
        run_name                     = CFG["run_name"] + "-medium",
        save_strategy                = "steps",
        save_steps                   = CFG["save_steps"],
        save_total_limit             = 2,
        push_to_hub                  = True,
        hub_model_id                 = HF_RL_REPO,
        hub_strategy                 = "checkpoint",
        seed                         = CFG["seed"],
        remove_unused_columns        = False,
    ),
    train_dataset = medium_ds,
    callbacks     = [monitor_callback],
)
print(f"Medium tier trainer ready: {len(medium_ds)} samples")
print(f"  {int((1-CFG['medium_math_frac'])*100)}% GSM 6-9 steps  +  "
      f"{int(CFG['medium_math_frac']*100)}% simple MATH-derived")
medium_trainer.train()
print("Medium tier training complete.")

Curriculum tier is: easy
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 2 to the `num_generations` of 6


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Medium tier trainer ready: 2000 samples
  100% GSM 6-9 steps  +  0% simple MATH-derived


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 4 x 1) = 24
 "-____-"     Trainable parameters = 4,616,192 of 1,548,330,496 (0.30% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'disable_compile', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Futu

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


...

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


  CURRICULUM PROMOTION at step 100
  Rolling accuracy exceeded 70% on EASY tier.
  Advancing to MEDIUM tier.
  ACTION: stop training cell and run Cell 14 (medium trainer).


═════════════════════════════════════════════════════════════════
  STEP 100 — MEDIUM TIER SAMPLE
  Rolling accuracy : 0.0%  (window=0/200)  promote at 70%
  Next tier        : hard
═════════════════════════════════════════════════════════════════
  QUESTION : If the mayor paid a total of $50,000 to two artists for painting 50 murals around the city, and Celina received $1,000 m
  GT ANSWER: 9800
  ─── MODEL OUTPUT (86 words) ───
  Let's assume Diego received x dollars.
Celina received $1,000 more than 4 times the amount Diego received, so she received 4x + $1,000.
The total amount paid to both artists is $50,000, so we can write the equation as x + (4x + $1,000) = $50,000.
Combining like terms, we get 5x + $1,000 = $50,000.
Subtracting $1,000 from both sides, we get 5x = $49,000.
Dividing both sides by 5, we get 

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen1b5_grpo_rl_medium/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen1b5_grpo_rl_medium/tokenizer_config.json.
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.mo

KeyboardInterrupt: 

## Cell 14b — Hard tier (run after medium promotes)

When the monitoring banner shows `CURRICULUM PROMOTION → HARD`, stop Cell 14a and run this cell.  
LR halved again — hard MATH problems have noisier rewards, small steps matter more.

In [ ]:
# ── 14b. Hard tier training ──────────────────────────────────────────────── ───
# Run when curriculum.tier == 'hard' 
print(f"Curriculum tier is: {curriculum.tier}")
assert curriculum.tier == "hard", \
    "Not yet promoted to hard — check rolling accuracy in WandB before running this cell"

gc.collect(); torch.cuda.empty_cache()

hard_trainer = GRPOTrainer( 
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = compute_rewards,
    args          = GRPOConfig(
        output_dir                   = CFG["output_dir"] + "_hard",
        num_generations              = CFG["grpo_group_size"],
        max_completion_length        = CFG["max_new_tokens"],
        beta                         = CFG["kl_coeff"],
        temperature                  = CFG["temperature"],
        num_train_epochs             = 1,
        per_device_train_batch_size  = CFG["per_device_batch"],
        gradient_accumulation_steps  = CFG["grad_accum"],
        learning_rate                = CFG["learning_rate"] * 0.5,  # lowest LR — hard tier is noisiest
        max_grad_norm                = CFG["max_grad_norm"],
        warmup_steps                 = 10,
        lr_scheduler_type            = "cosine",
        fp16                         = CFG["fp16"],
        bf16                         = CFG["bf16"],
        optim                        = "adamw_8bit",
        gradient_checkpointing       = True,
        gradient_checkpointing_kwargs= {"use_reentrant": False},
        logging_steps                = 1,
        logging_first_step           = True,
        report_to                    = "none",
        run_name                     = CFG["run_name"] + "-hard",
        save_strategy                = "steps",
        save_steps                   = CFG["save_steps"],
        save_total_limit             = 2,
        push_to_hub                  = True,
        hub_model_id                 = HF_RL_REPO,
        hub_strategy                 = "checkpoint",
        seed                         = CFG["seed"],
        remove_unused_columns        = False,
    ),
    train_dataset = hard_ds,
    callbacks     = [monitor_callback],
)
print(f"Hard tier trainer ready: {len(hard_ds)} samples")
print("  MATH-derived problems, 9+ reasoning steps")
hard_trainer.train()
print("Hard tier training complete.")

## Cell 15 — Save adapter + post-training inference test

In [ ]:
# ── 15. Save RL adapter ───────────────────────────────────────────────────────
rl_adapter_dir = os.path.join(CFG["output_dir"], "rl_lora_adapter")
os.makedirs(rl_adapter_dir, exist_ok=True)

model.save_pretrained(rl_adapter_dir)
tokenizer.save_pretrained(rl_adapter_dir)
print(f"RL LoRA adapter saved to: {rl_adapter_dir}")

# ── Post-training inference test ──────────────────────────────────────────────
FastLanguageModel.for_inference(model)

test_questions = [
    "Janet's ducks lay 16 eggs per day. She eats 3 for breakfast and uses 4 to bake muffins. She sells the rest at the market for $2 each. How much does she make per day?",
    "If a train travels at 60 km/h for 2.5 hours, how far does it travel?",
    "A store sells notebooks for $3 each. If Sarah buys 7 notebooks and pays with a $25 bill, how much change does she get?",
]

print("\n" + "═" * 65)
print("  POST-TRAINING INFERENCE TEST (RL model)")
print("═" * 65)

for q in test_questions:
    prompt = make_prompt(q)
    enc    = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=CFG["max_seq_len"] // 2).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens = CFG["max_new_tokens"],
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
        )
    response = tokenizer.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    pred_ans = extract_answer(response)
    print(f"\nQ: {q}")
    print(f"A: {response}")
    print(f"Extracted answer: {pred_ans}")
    print("-" * 65)

wandb.finish()
print("\nDone. WandB run closed.")

---

## Tradeoffs and design decisions

### What was changed from the SFT notebook and why

| Decision | SFT notebook | This RL notebook | Reason |
|---|---|---|---|
| LoRA rank | r=16 | r=4 | RL reshapes distribution, not injecting knowledge. 4x fewer params, less forgetting. |
| `load_in_4bit` | False | **False** | Loading adapter repo with `True` leaves SFT LoRA attached — `get_peft_model()` raises `TypeError`. `False` auto-merges SFT adapter into dense fp16 base, giving a clean model for RL LoRA injection. Also eliminates the `c10::Half != float` dtype mismatch. |
| `dtype` | None (auto) | `torch.float16` | Explicit fp16 for T4. Avoids silent bfloat16 fallback which caused precision issues on CC=7.5 GPUs. |
| `prompt` column | not needed (SFTTrainer uses `text`) | **required** | GRPOTrainer does `x["prompt"]` internally. Missing column → `KeyError` at step 0. Added via `.map()` after `Dataset.from_list()`. |
| Curriculum tiers | n/a | easy → medium → hard | 3-tier: GSM ≤5 steps / GSM 6-9 + simple MATH / MATH 9+ steps. Tier stored in `CurriculumState.tier`, promoted via `next_tier()`. |
| `max_seq_len` | 2048 | 512 | Shorter sequences = more rollout headroom. GSM8K problems don't need 2048 tokens. |
| Learning rate | 2e-4 | 5e-7 → 3.5e-7 → 2.5e-7 | Decreasing per tier: hard MATH rewards are noisier, smaller steps matter more. |
| `max_grad_norm` | 1.0 | 0.1 | Tighter clip for RL stability. GRPO gradients spike more than SFT. |
| Dataset size | 120k | 1.5k × 3 tiers | One full pass through 1.5k at ~30s/step ≈ 12.5 hours — one Kaggle session per tier. |

### Honest limitation: curriculum hot-swapping

GRPOTrainer takes the dataset at `__init__` time. Mid-training dataset hot-swap is not supported
by TRL's trainer loop without subclassing. The practical workaround: the callback prints a
`CURRICULUM PROMOTION` banner and logs `curriculum/tier` to WandB when 70% accuracy is hit.
You stop the current training cell and run the next tier cell (14a or 14b) manually.
One-line assertion at the top of each tier cell guards against running out of order.

### WandB metrics to watch

- `reward` — mean group reward per step. Should trend upward slowly.
- `reward_std` — variance within GRPO group. If this collapses to ~0, GRPO has stalled.
- `kl` — should stay below 0.5. Spike = model diverging from SFT reference.
- `curriculum/tier` — 0/1/2 for easy/medium/hard. Jumps are promotion events.
- `curriculum/rolling_accuracy` — your 70% gate. Watch this approach the threshold.
- `monitor/sample_reward` — greedy sample reward every 25 steps. Good sanity check.
- `monitor/token_count` — should stay below 400 if length penalty is working.

### If OOM during training

1. Reduce `grpo_group_size` from 6 to 4 — less rollout memory at generation time
2. Reduce `max_new_tokens` from 300 to 200
3. These two together save ~2-3 GB at the generation phase peak


In [ ]:
billo bagge billaya da ki karegi